# Mastercard Data Quest 2026 — Hidden Entrepreneur Detection (Genius Level)
## 10 Behavioral Paradoxes + Cross-border Sourcing + AUC-ROC + Target Product

**Objective:** Given ~13M credit-card transactions (105K cards), classify which
consumer *card_number*s behave like business cards, revealing hidden entrepreneurs.


## Requirements — Install dependencies (Reproducibility 5%)

In [ ]:
# Run this cell ONCE to install all dependencies
import sys
%pip install pandas pyarrow numpy catboost scikit-learn matplotlib shap nbformat -q
print("All dependencies installed.")
print("Python:", sys.version)


## Imports

In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool, cv as catboost_cv
from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report, confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from IPython.display import Image, display

import shap

warnings.filterwarnings("ignore")
DATA_DIR = Path(".")
OUTPUT_DIR = Path(".")
print("All imports successful.")


All imports successful.


## Step 1 — Data Loading

In [3]:
biz = pd.read_parquet(
    DATA_DIR / "business_cards_MDQ.parquet",
    columns=[
        "transaction_date", "transaction_timestamp", "transaction_amount_kzt",
        "mcc", "merchant_id", "channel", "card_number", "tokenized", "is_recurring",
        "country",
    ],
)
con = pd.read_parquet(
    DATA_DIR / "consumer_cards_MDQ.parquet",
    columns=[
        "transaction_date", "transaction_timestamp", "transaction_amount_kzt",
        "mcc", "merchant_id", "channel", "card_number", "tokenized", "is_recurring",
        "country",
    ],
)
mer_ref = pd.read_parquet(DATA_DIR / "merchants_reference.parquet")

print(f"Business cards : {len(biz):,} rows  |  {biz['card_number'].nunique():,} unique cards")
print(f"Consumer cards : {len(con):,} rows  |  {con['card_number'].nunique():,} unique cards")
print(f"Merchant ref   : {len(mer_ref):,} rows  |  {mer_ref['merchant_id'].nunique():,} merchants")


Business cards : 2,997,593 rows  |  25,000 unique cards
Consumer cards : 9,832,487 rows  |  80,000 unique cards
Merchant ref   : 2,165 rows  |  2,165 merchants


## Step 2 — Label Assignment & Concatenation + Merchant Merge

In [4]:
biz["label"] = 1
con["label"] = 0

cols = [
    "transaction_date", "transaction_timestamp", "transaction_amount_kzt",
    "mcc", "merchant_id", "channel", "card_number", "tokenized", "is_recurring",
    "country", "label",
]
df = pd.concat([biz[cols], con[cols]], ignore_index=True)
del biz, con

df = df.merge(
    mer_ref[["merchant_id", "mcc", "recurring_capable", "merchant_name", "merchant_country"]],
    on=["merchant_id", "mcc"],
    how="left",
)
del mer_ref

print(f"Combined: {len(df):,} rows  |  {df['card_number'].nunique():,} cards")
print(f"Label distrib: 0 = {(df['label']==0).sum():,}  |  1 = {(df['label']==1).sum():,}")


Combined: 12,830,080 rows  |  105,000 cards
Label distrib: 0 = 9,832,487  |  1 = 2,997,593


## Step 3 — MCC Category Definitions

In [5]:
BUSINESS_MCCS: set = {
    "7311", "7372", "5968", "4816", "7379", "5734",
    "7392", "7399", "8931", "8111", "8911", "7361",
    "7311", "4812", "4814",
    "5021", "5044", "5045", "5046", "5111", "5943",
    "5099", "5199", "5131", "5137", "5139", "5169", "5172", "5193",
    "5039", "5065", "5072", "5085", "5211", "5231",
    "4214", "4215", "4225", "4011",
    "4511", "7011", "4722", "4723",
    "7394", "7512",
    "6381", "7321",
    "7333", "7338", "7298", "7299",
    "5199", "5261",
    "5122", "5172",
    "5200",
}

WHOLESALE_PROD_MCCS: set = {
    "5021", "5039", "5044", "5045", "5046", "5065", "5072", "5085",
    "5099", "5111", "5122", "5131", "5137", "5139", "5169", "5172",
    "5193", "5199", "5200", "5211", "5231",
}

LOGISTICS_MCCS: set = {
    "4214", "4215", "4225", "4011", "4457", "7511",
}

FUEL_MCCS: set = {"5541", "5542", "5983"}

SAAS_AD_MCCS: set = {
    "7311", "7372", "5968", "4816", "4812", "4814", "7379", "5734",
}

print(f"Business MCCs: {len(BUSINESS_MCCS)}")
print(f"Wholesale/Prod MCCs: {len(WHOLESALE_PROD_MCCS)}")
print(f"Logistics MCCs: {len(LOGISTICS_MCCS)}")
print(f"Fuel MCCs: {len(FUEL_MCCS)}")
print(f"SaaS/Ad MCCs: {len(SAAS_AD_MCCS)}")


Business MCCs: 53
Wholesale/Prod MCCs: 21
Logistics MCCs: 6
Fuel MCCs: 3
SaaS/Ad MCCs: 8


## Step 4 — 10 Behavioral Paradoxes + Cross-border

### 1. Clockwork Buyer → CV of inter-purchase intervals
### 2. Off-Hours Operator → fraction of biz-MCC txns outside 9-19
### 3. Expense Ratio Inversion → wholesale spend share
### 4. Token Wholesale → high-value tokenised wholesale
### 5. Supplier Fingerprint (NEW) → vendor concentration (top-3 merchants)
### 6. Last-Mile Echo (NEW) → wholesale→logistics time lag
### 7. Round-Trip Cash Flow — Redefined (NEW) → spending burst periodicity
### 8. Inventory Pulse (NEW) → small-to-large wholesale txn ratio
### 9. Multi-Vendor Loyalty Paradox (NEW) → B2B volume without recurring
### 10. Channel Schizophrenia (NEW) → channel alternation rate
### 🌍 Cross-border Sourcing (NEW) → international SaaS/Ads txns


### 4a. Clockwork Buyer

In [6]:
print("Feature 4a — Clockwork Buyer ...")
df.sort_values(["card_number", "mcc", "transaction_timestamp"], inplace=True)
df["interval_hours"] = df.groupby(["card_number", "mcc"])["transaction_timestamp"].diff().dt.total_seconds() / 3600.0
cv = df.dropna(subset=["interval_hours"]).groupby(["card_number", "mcc"])["interval_hours"].agg(lambda x: x.std()/x.mean() if x.mean()>1e-9 else np.nan).rename("cv").reset_index()
clockwork_cv = cv.groupby("card_number")["cv"].mean().rename("clockwork_cv_mean")
clockwork_cnt = cv.dropna(subset=["cv"]).groupby("card_number")["cv"].count().rename("clockwork_mcc_count")
df.drop(columns=["interval_hours"], inplace=True)
del cv
print(f"  -> {len(clockwork_cv):,} cards")


Feature 4a — Clockwork Buyer ...
  -> 105,000 cards


### 4b. Off-Hours Operator

In [7]:
print("Feature 4b — Off-Hours Operator ...")
df["txn_hour"] = df["transaction_timestamp"].dt.hour
df["is_business_mcc"] = df["mcc"].isin(BUSINESS_MCCS)
df["is_off_hours"] = (df["txn_hour"] < 9) | (df["txn_hour"] >= 19)
biz_mcc_txns = df[df["is_business_mcc"]].groupby("card_number").size()
off_hours_biz = df[df["is_business_mcc"] & df["is_off_hours"]].groupby("card_number").size()
off_hours_ratio = (off_hours_biz / biz_mcc_txns).rename("off_hours_ratio").fillna(0.0)
off_hours_total = df[df["is_off_hours"]].groupby("card_number").size().rename("off_hours_total_count")
total_txns = df.groupby("card_number").size().rename("total_txns")
overall_off_hours_ratio = (off_hours_total / total_txns).rename("overall_off_hours_ratio")
print(f"  -> {len(off_hours_ratio):,} cards")


Feature 4b — Off-Hours Operator ...
  -> 104,600 cards


### 4c. Expense Ratio Inversion

In [8]:
print("Feature 4c — Expense Ratio Inversion ...")
total_spend = df.groupby("card_number")["transaction_amount_kzt"].sum().rename("total_spend_kzt")
wholesale_spend = df[df["mcc"].isin(WHOLESALE_PROD_MCCS)].groupby("card_number")["transaction_amount_kzt"].sum().rename("wholesale_spend_kzt")
expense_ratio = (wholesale_spend / total_spend).rename("wholesale_spend_ratio").fillna(0.0)
wholesale_spend_log = np.log1p(wholesale_spend).rename("wholesale_spend_log")
print(f"  -> {len(expense_ratio):,} cards")


Feature 4c — Expense Ratio Inversion ...
  -> 105,000 cards


### 4d. Token Wholesale

In [9]:
print("Feature 4d — Token Wholesale ...")
tw = df[df["mcc"].isin(WHOLESALE_PROD_MCCS) & (df["tokenized"] == True)].copy()
if len(tw) > 0:
    p90 = tw[tw["label"] == 1]["transaction_amount_kzt"].quantile(0.90)
    print(f"  -> 90th pctile (business): {p90:,.0f} KZT")
else: p90 = 0
token_flag = tw[tw["transaction_amount_kzt"] > p90].groupby("card_number").size().gt(0).astype(int).rename("token_wholesale_flag")
token_cnt = tw[tw["transaction_amount_kzt"] > p90].groupby("card_number").size().rename("token_wholesale_count")
del tw
print(f"  -> {token_flag.sum():,} cards with high-value tokenized wholesale")


Feature 4d — Token Wholesale ...
  -> 90th pctile (business): 470,088 KZT
  -> 19,368 cards with high-value tokenized wholesale


### 4e. Baseline Aggregations

In [10]:
print("Feature 4e — Baseline ...")
card_agg = df.groupby("card_number").agg(
    amount_mean=("transaction_amount_kzt","mean"), amount_std=("transaction_amount_kzt","std"),
    amount_sum=("transaction_amount_kzt","sum"), amount_max=("transaction_amount_kzt","max"),
    n_unique_mcc=("mcc","nunique"), n_unique_merchants=("merchant_id","nunique"),
    n_online=("channel",lambda x:(x=="online").sum()), n_pos=("channel",lambda x:(x=="POS").sum()),
    tokenized_txn_count=("tokenized","sum"), recurring_count=("is_recurring","sum"),
)
card_agg["online_ratio"] = card_agg["n_online"]/(card_agg["n_online"]+card_agg["n_pos"])
card_agg["tokenized_ratio"] = card_agg["tokenized_txn_count"]/total_txns
card_agg["n_unique_mcc_log"] = np.log1p(card_agg["n_unique_mcc"])
print(f"  -> {len(card_agg):,} cards")


Feature 4e — Baseline ...
  -> 105,000 cards


### 5. Supplier Fingerprint (Vendor Concentration)

In [11]:
print("Feature 5 — Supplier Fingerprint ...")
ms = df.groupby(["card_number","merchant_id"])["transaction_amount_kzt"].sum().reset_index()
total_per_card = ms.groupby("card_number")["transaction_amount_kzt"].transform("sum")
ms["share"] = ms["transaction_amount_kzt"]/total_per_card
ms = ms.sort_values(["card_number","share"], ascending=[True,False])
vendor_concentration = ms.groupby("card_number").head(3).groupby("card_number")["share"].sum().rename("vendor_concentration")
def _gini(g):
    a = g["transaction_amount_kzt"].sort_values().values
    if len(a)==0 or a.sum()==0: return 0.0
    cumsum = np.cumsum(a); return (2*cumsum.sum()/a.sum()-len(a)-1)/len(a)
merchant_gini = ms.groupby("card_number").apply(_gini, include_groups=False).rename("merchant_gini")
del ms
print(f"  -> done")


Feature 5 — Supplier Fingerprint ...
  -> done


### 6. Last-Mile Echo (Wholesale → Logistics Lag)

In [12]:
print("Feature 6 — Last-Mile Echo ...")
dw = df[df["mcc"].isin(WHOLESALE_PROD_MCCS)].groupby(["card_number","transaction_date"])["transaction_amount_kzt"].sum().reset_index()
dw.columns = ["card_number","date","wholesale_amt"]
dl = df[df["mcc"].isin(LOGISTICS_MCCS|FUEL_MCCS)].groupby(["card_number","transaction_date"])["transaction_amount_kzt"].sum().reset_index()
dl.columns = ["card_number","date","logistics_amt"]
dm = dw.merge(dl,on=["card_number","date"],how="outer").fillna(0).sort_values(["card_number","date"]).reset_index(drop=True)
dm["both"]=(dm["wholesale_amt"]>0)&(dm["logistics_amt"]>0)
dm["any"]=(dm["wholesale_amt"]>0)|(dm["logistics_amt"]>0)
wholesale_logistics_cross_ratio = dm[dm["any"]].groupby("card_number")["both"].mean().rename("wholesale_to_logistics_count")
log_dates = {cn: g["date"].values.astype("datetime64[D]") for cn, g in dl.groupby("card_number",sort=False)}
def _echo(g):
    cn=g.name; w=g.loc[g["wholesale_amt"]>0,"date"].values.astype("datetime64[D]")
    l=log_dates.get(cn,np.array([],dtype="datetime64[D]")); 
    if len(w)==0 or len(l)==0: return 0.0
    i=np.clip(np.searchsorted(l,w,side="left"),0,len(l)-1); la=(l[i]-w).astype(int)
    v=(la>=0)&(la<=7); return float(la[v].mean()) if v.sum()>0 else 0.0
wholesale_to_logistics_lag = dm.groupby("card_number",sort=False).apply(_echo,include_groups=False).rename("wholesale_to_logistics_lag")
del dw,dl,dm,log_dates
print(f"  -> done")


Feature 6 — Last-Mile Echo ...
  -> done


### 7. Round-Trip Cash Flow (Spending Burst Periodicity)

In [13]:
print("Feature 7 — Round-Trip CF ...")
daily_spend = df.groupby(["card_number","transaction_date"])["transaction_amount_kzt"].sum().reset_index()
def _burst(g):
    a=g["transaction_amount_kzt"]
    if len(a)<5: return np.nan
    t=a.quantile(0.95); b=g[a>=t]["transaction_date"].sort_values()
    if len(b)<2: return np.nan
    d=b.diff().dt.days.dropna(); return float(d.std()) if len(d)>=2 else np.nan
spending_burst_periodicity = daily_spend.groupby("card_number",sort=False).apply(_burst,include_groups=False).rename("spending_burst_periodicity")
del daily_spend
print(f"  -> done")


Feature 7 — Round-Trip CF ...
  -> done


### 8. Inventory Pulse

In [14]:
print("Feature 8 — Inventory Pulse ...")
def _pulse(g):
    w=g[g["mcc"].isin(WHOLESALE_PROD_MCCS)]["transaction_amount_kzt"]
    if len(w)<5: return np.nan
    p30,p70=w.quantile(0.30),w.quantile(0.70); s,l=(w<=p30).sum(),(w>=p70).sum()
    return s/l if l>0 else np.nan
massive_to_small_ratio = df.groupby("card_number",sort=False).apply(_pulse,include_groups=False).rename("massive_to_small_ratio")
print(f"  -> done")


Feature 8 — Inventory Pulse ...
  -> done


### 9. Multi-Vendor Loyalty Paradox

In [15]:
print("Feature 9 — Multi-Vendor Loyalty ...")
b2b_spend = df[df["mcc"].isin(BUSINESS_MCCS)].groupby("card_number")["transaction_amount_kzt"].sum()
recurring_ratio = df.groupby("card_number")["is_recurring"].mean()
b2b_volume_no_recurring = (b2b_spend*(1-recurring_ratio)).rename("b2b_volume_no_recurring").fillna(0.0)
txns_per_merchant = (total_txns/df.groupby("card_number")["merchant_id"].nunique()).rename("txns_per_merchant")
print(f"  -> done")


Feature 9 — Multi-Vendor Loyalty ...
  -> done


### 10. Channel Schizophrenia

In [16]:
print("Feature 10 — Channel Schizophrenia ...")
ds = df.sort_values(["card_number","transaction_timestamp"])
ds["prev_ch"]=ds.groupby("card_number")["channel"].shift(1)
ds["switch"]=(ds["channel"]!=ds["prev_ch"])&ds["prev_ch"].notna()
channel_alternation_rate = ds.groupby("card_number")["switch"].mean().rename("channel_alternation_rate")
def _ent(g): c=g.value_counts(); p=c/c.sum(); return float(-(p*np.log2(p+1e-10)).sum())
channel_entropy = ds.groupby("card_number")["channel"].apply(_ent).rename("channel_entropy")
del ds
print(f"  -> done")


Feature 10 — Channel Schizophrenia ...
  -> done


### 🌍 Cross-border Sourcing

In [17]:
print("Feature 11 — Cross-border ...")
df["is_cross_border"]=(df["merchant_country"]!=df["country"])&df["merchant_country"].notna()
df["is_cb_saas_ad"]=df["is_cross_border"]&df["mcc"].isin(SAAS_AD_MCCS)
cb_ratio=df.groupby("card_number")["is_cross_border"].mean().rename("cb_ratio")
cb_saas_ad_ratio=df[df["mcc"].isin(SAAS_AD_MCCS)].groupby("card_number")["is_cross_border"].mean().rename("cb_saas_ad_ratio")
cb_spend=df[df["is_cross_border"]].groupby("card_number")["transaction_amount_kzt"].sum()
cb_spend_share=(cb_spend/total_spend).rename("cb_spend_share").fillna(0.0)
cb_saas_ad_count=df[df["is_cb_saas_ad"]].groupby("card_number").size().rename("cb_saas_ad_count")
df.drop(columns=["txn_hour","is_business_mcc","is_off_hours","is_cross_border","is_cb_saas_ad"],inplace=True,errors="ignore")
print(f"  -> done")


Feature 11 — Cross-border ...
  -> done


## Step 5 — Feature Assembly (35 features)

In [18]:
labels = df[["card_number","label"]].drop_duplicates("card_number").set_index("card_number")["label"]
feat = pd.DataFrame(index=labels.index)
for s in [clockwork_cv,clockwork_cnt,off_hours_ratio,overall_off_hours_ratio,off_hours_total,expense_ratio,wholesale_spend_log,token_flag,token_cnt]:
    feat[s.name]=s
for c in ["total_txns","amount_mean","amount_std","amount_sum","amount_max","n_unique_mcc_log","n_unique_mcc","n_unique_merchants","online_ratio","tokenized_ratio","n_online","n_pos"]:
    feat[c]=card_agg[c]
for s in [vendor_concentration,merchant_gini,wholesale_to_logistics_lag,wholesale_logistics_cross_ratio,spending_burst_periodicity,massive_to_small_ratio,b2b_volume_no_recurring,txns_per_merchant,channel_alternation_rate,channel_entropy,cb_ratio,cb_saas_ad_ratio,cb_spend_share,cb_saas_ad_count]:
    feat[s.name]=s
# Fill NaNs
feat["clockwork_cv_mean"]=feat["clockwork_cv_mean"].fillna(feat["clockwork_cv_mean"].median())
for c in ["clockwork_mcc_count","off_hours_ratio","overall_off_hours_ratio","off_hours_total_count","wholesale_spend_ratio","wholesale_spend_log","token_wholesale_flag","token_wholesale_count","online_ratio","tokenized_ratio","vendor_concentration","merchant_gini","b2b_volume_no_recurring","cb_ratio","cb_saas_ad_ratio","cb_spend_share"]:
    feat[c]=feat[c].fillna(0)
feat["amount_std"]=feat["amount_std"].fillna(0.0)
feat["wholesale_to_logistics_lag"]=feat["wholesale_to_logistics_lag"].fillna(168.0)
feat["spending_burst_periodicity"]=feat["spending_burst_periodicity"].fillna(feat["spending_burst_periodicity"].median())
feat["massive_to_small_ratio"]=feat["massive_to_small_ratio"].fillna(feat["massive_to_small_ratio"].median())
feat["txns_per_merchant"]=feat["txns_per_merchant"].fillna(1.0)
feat["channel_alternation_rate"]=feat["channel_alternation_rate"].fillna(0.0)
feat["channel_entropy"]=feat["channel_entropy"].fillna(0.0)
feat["cb_saas_ad_count"]=feat["cb_saas_ad_count"].fillna(0).astype(int)
feat=feat.dropna(); labels=labels.loc[feat.index]
print(f"Matrix: {feat.shape[0]:,} cards x {feat.shape[1]} features")
print(f"Columns: {list(feat.columns)}")


KeyError: 'total_txns'

## Step 6 — Train / Test Split (80/20 stratified)

In [ ]:
feature_names=list(feat.columns)
X_train,X_test,y_train,y_test = train_test_split(feat.values,labels.values,test_size=0.2,random_state=42,stratify=labels.values)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")


## Step 7 — CatBoost Training

**eval_metric = AUC** (not Accuracy — avoids class imbalance bias)

In [ ]:
model = CatBoostClassifier(
    iterations=500, learning_rate=0.1, depth=6,
    loss_function="Logloss", eval_metric="AUC",
    auto_class_weights="Balanced", early_stopping_rounds=50,
    verbose=50, random_seed=42, task_type="CPU", allow_writing_files=False,
)
model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)
print("\nTraining complete.")


## Step 7b — 5-Fold Cross-Validation (AUC-ROC)

In [ ]:
cv_model = CatBoostClassifier(
    iterations=200, learning_rate=0.1, depth=6,
    loss_function="Logloss", eval_metric="AUC",
    auto_class_weights="Balanced", random_seed=42,
    task_type="CPU", allow_writing_files=False, verbose=0,
)
cv_results = catboost_cv(Pool(feat.values,labels.values,feature_names=feature_names),cv_model.get_params(),fold_count=5,stratified=True,early_stopping_rounds=30,verbose_eval=False)
if "test-AUC-mean" in cv_results.columns:
    print(f"CV AUC-ROC mean: {cv_results['test-AUC-mean'].iloc[-1]:.4f}")
    print(f"CV AUC-ROC std:  {cv_results['test-AUC-std'].iloc[-1]:.4f}")
else:
    print(f"CV columns: {cv_results.columns.tolist()}")


## Step 8 — Evaluation

### PRIMARY: AUC-ROC
### SECONDARY: Confusion Matrix (jury requirement) — displayed below

In [ ]:
print("="*60)
print("PRIMARY — AUC-ROC".center(60))
print("="*60)
y_pred=model.predict(X_test); y_proba=model.predict_proba(X_test)[:,1]
auc=roc_auc_score(y_test,y_proba)
print(f"AUC-ROC: {auc:.4f}")

print("\n"+"="*60)
print("CONFUSION MATRIX (secondary)".center(60))
print("="*60)
cm=confusion_matrix(y_test,y_pred)
print(f"{'':>10} {'Pred 0':>8} {'Pred 1':>8}")
print(f"{'Actual 0':>10} {cm[0,0]:>8} {cm[0,1]:>8}")
print(f"{'Actual 1':>10} {cm[1,0]:>8} {cm[1,1]:>8}")
print("\n"+classification_report(y_test,y_pred,target_names=["Consumer","Business"]))


### Confusion Matrix Visualization (saved + displayed inline)

In [ ]:
ConfusionMatrixDisplay.from_estimator(model,X_test,y_test,display_labels=["Consumer","Business"],cmap="Blues")
plt.title(f"Confusion Matrix — Hidden Entrepreneurs (AUC={auc:.4f})")
plt.savefig(OUTPUT_DIR/"confusion_matrix.png",dpi=150,bbox_inches="tight")
plt.show()
print("Saved -> confusion_matrix.png")
# Display the saved PNG for jury visibility
display(Image(filename=str(OUTPUT_DIR/"confusion_matrix.png")))


## Step 9 — Feature Importance (CatBoost) — displayed below

In [ ]:
fi=pd.DataFrame({"feature":feature_names,"importance":model.feature_importances_}).sort_values("importance",ascending=False)
print("\n"+"="*60)
print("FEATURE IMPORTANCE".center(60))
print("="*60)
for i,(_,r) in enumerate(fi.iterrows(),1):
    bar="█"*int(r["importance"]/2)
    print(f"  {i:2d}. {r['feature']:<28s} {r['importance']:6.2f}%  {bar}")

plt.figure(figsize=(10,6)); top=fi.head(15)
plt.barh(range(len(top)),top["importance"].values,color="steelblue")
plt.yticks(range(len(top)),top["feature"].values); plt.gca().invert_yaxis()
plt.xlabel("Importance (%)"); plt.title("Top 15 Feature Importances — CatBoost")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"feature_importance.png",dpi=150,bbox_inches="tight")
plt.show()
print("Saved -> feature_importance.png")
display(Image(filename=str(OUTPUT_DIR/"feature_importance.png")))


## Step 10 — SHAP Explainability

In [ ]:
print("Computing SHAP values (500-sample subset) ...")
test_df=pd.DataFrame(X_test[:500],columns=feature_names)
explainer=shap.TreeExplainer(model); shap_values=explainer.shap_values(test_df)
plt.figure(figsize=(12,8))
shap.summary_plot(shap_values,test_df,show=False)
plt.title("SHAP Summary — Impact on Business Classification")
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"shap_summary.png",dpi=150,bbox_inches="tight"); plt.show()
print("Saved -> shap_summary.png")
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values,test_df,plot_type="bar",show=False)
plt.title("SHAP Mean |Value| — Feature Importance (Bar)")
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"shap_bar.png",dpi=150,bbox_inches="tight"); plt.show()
print("Saved -> shap_bar.png")


## Step 11 — Top 100 Hidden Entrepreneurs + Target Product

**Business Logic:**
- **B2B Cashback** → high wholesale_spend_ratio
- **Working Capital Loan** → high clockwork_cv_mean
- **Merchant Account** → high channel_alternation_rate
- **Business Credit Card** → default

In [ ]:
consumer_mask=labels==0; consumer_feat=feat.loc[consumer_mask]
consumer_cards=consumer_feat.index.values
consumer_probas=model.predict_proba(consumer_feat.values)[:,1]
idx=np.argsort(consumer_probas)[-100:][::-1]
top_cards=consumer_cards[idx]; top_scores=consumer_probas[idx]
fc=feat.loc[consumer_mask]
cv_th=fc["clockwork_cv_mean"].median(); ws_th=fc["wholesale_spend_ratio"].median(); sch_th=fc["channel_alternation_rate"].median()
top_cv=fc.loc[top_cards,"clockwork_cv_mean"]; top_ws=fc.loc[top_cards,"wholesale_spend_ratio"]; top_sch=fc.loc[top_cards,"channel_alternation_rate"]
prods=[]
for i in range(100):
    if top_ws.iloc[i]>ws_th: prods.append("B2B Cashback")
    elif top_cv.iloc[i]>cv_th: prods.append("Working Capital Loan")
    elif top_sch.iloc[i]>sch_th: prods.append("Merchant Account")
    else: prods.append("Business Credit Card")
top_df=pd.DataFrame({"card_number":top_cards,"business_probability":top_scores,"rank":range(1,101),"recommended_product":prods})
print("\nTop 10:"); print(top_df.head(10).to_string(index=False))
top_df.to_csv(OUTPUT_DIR/"top100_hidden_entrepreneurs.csv",index=False)
print(f"\nSaved -> top100_hidden_entrepreneurs.csv ({len(top_df)} records)")
for q in [0,25,50,75,90,95,99,100]:
    print(f"  P{q:>3}: {np.percentile(consumer_probas,q):.4f}")
print(f"  Cards prob>0.5: {(consumer_probas>0.5).sum():,}/{len(consumer_probas):,}")
print(f"\nProduct distribution:\n{top_df['recommended_product'].value_counts().to_string()}")


## Step 12 — Deliverables Checklist

In [ ]:
import os
files=["confusion_matrix.png","feature_importance.png","shap_summary.png","shap_bar.png","top100_hidden_entrepreneurs.csv"]
print("\n"+"="*60); print("DELIVERABLES".center(60)); print("="*60)
for f in files:
    p=OUTPUT_DIR/f; e=os.path.exists(p)
    print(f"  {'OK' if e else 'MISSING':8} {f:<40s} {os.path.getsize(p) if e else 0:>8,} bytes")
print("\nAll deliverables generated. Ready for submission.")
